In [1]:
import pandas as pd
import numpy as np
import ast

df = pd.read_csv("../data/raw/tweets_ai.csv")
df["id"] = df["id"].astype("int64").astype("string")
df["conversation_id"] = df["conversation_id"].astype("int64").astype("string")
df["created_at"] = pd.to_datetime(
    df["created_at"].str.replace(r"\s[A-Z]{3}$", "", regex=True)
)
df["date"] = pd.to_datetime(df["date"])

df.head()

,id,conversation_id,created_at,date,time,timezone,tweet,language,urls,photos,replies_count,retweets_count,likes_count,hashtags,cashtags,retweet,quote_url,video,thumbnail
0,826580756675694720,826580756675694720,2017-01-31 18:59:44,2017-01-31,18:59:44,-400,What topic could spark human dialogue on AI? A...,en,['https://blog.hubspot.com/marketing/artificia...,[],0,0,0,[],[],False,NaN,0,NaN
1,826580719921152000,826580719921152000,2017-01-31 18:59:35,2017-01-31,18:59:35,-400,I think #ArtificialIntelligence will augment h...,en,['http://bit.ly/2bWb1ly'],[],0,4,1,"['artificialintelligence', 'ai', 'chasmwaxing']",[],False,NaN,0,NaN
2,826580631366700928,826580631366700928,2017-01-31 18:59:14,2017-01-31,18:59:14,-400,What Managers Need to Know About Artificial In...,en,['http://mitsmr.com/2klQ17D'],[],0,1,0,[],[],False,NaN,0,NaN
3,826580578954780672,826580578954780672,2017-01-31 18:59:01,2017-01-31,18:59:01,-400,4 ways Artificial Intelligence changes the gam...,en,['http://bit.ly/2koUCIB'],[],0,0,0,['thenextweb'],[],False,NaN,0,NaN
4,826580577457287040,826580577457287040,2017-01-31 18:59:01,2017-01-31,18:59:01,-400,Artificial Intelligence can Shape the Future o...,en,['https://www.youtube.com/watch?v=NQefTsrSjJY'],[],0,0,0,[],[],False,NaN,0,NaN


#### Convertimos texto ("['a', 'b']" o "[]") en listas reales de Python

In [2]:
def a_lista(x):

    if isinstance(x, list):
        return x

    if isinstance(x, np.ndarray): #si es una array de numpy lo pasamos a lista
        return x.tolist()

    if x is None:
        return []

    if pd.isna(x):
        return []

    if isinstance(x, str):
        texto = x.strip() # quitamos espacios 

        if texto == "" or texto.lower() == "nan":
            return []

        try:
            valor = ast.literal_eval(texto) # "['a','b']" -> ['a','b']

            if isinstance(valor, list):
                return valor
            return [valor]

        except (ValueError, SyntaxError):
            return []
    return []


In [3]:
columnas_lista = ["urls", "photos", "hashtags", "cashtags"] #Aplicamos la funcion anterior a las columnas del dataset en la que ocurre esto

for col in columnas_lista:
    df[col] = df[col].apply(a_lista)

In [4]:
print(df.loc[1, "hashtags"], type(df.loc[1, "hashtags"])) #check rapido
print(df.loc[1, "urls"], type(df.loc[1, "urls"]))
print(df.loc[0, "photos"], type(df.loc[0, "photos"]))

['artificialintelligence', 'ai', 'chasmwaxing'] <class 'list'>
['http://bit.ly/2bWb1ly'] <class 'list'>
[] <class 'list'>


#### Extraer los dominios de las URLs

In [5]:
from urllib.parse import urlparse #descompone una URL en partes

def get_dominio(url):
    try:
        dominio = urlparse(url).netloc.lower()

        if dominio.startswith("www."):
            dominio = dominio[4:] #nos quedamos solo desde el www. en adelante
        return dominio

    except Exception:
        return None

In [6]:
def extraer_dominios(lista_urls): #devuelve lista dominios unicos
    dominios = []
    
    for url in lista_urls:
        dominio = get_dominio(url)
        if dominio is not None:
            dominios.append(dominio)

    dominios_unicos = []
    vistos = set() #impide que haya duplicados
    
    for d in dominios:
        if d not in vistos:
            dominios_unicos.append(d)
            vistos.add(d)

    return dominios_unicos
    
df["dominios_url"] = df["urls"].apply(extraer_dominios)

In [7]:
df.loc[:10, ["urls", "dominios_url"]]

,urls,dominios_url
0,[https://blog.hubspot.com/marketing/artificial...,[blog.hubspot.com]
1,[http://bit.ly/2bWb1ly],[bit.ly]
2,[http://mitsmr.com/2klQ17D],[mitsmr.com]
3,[http://bit.ly/2koUCIB],[bit.ly]
4,[https://www.youtube.com/watch?v=NQefTsrSjJY],[youtube.com]
5,[http://buff.ly/2kL0sVB],[buff.ly]
6,[http://bit.ly/2kO88U7],[bit.ly]
7,[http://upflow.co/l/45Ug],[upflow.co]
8,[http://buff.ly/2kJMRht],[buff.ly]
9,[https://twitter.com/tommyxtopher/status/82652...,[twitter.com]


#### Clasificar los dominios por tipo de fuente

In [8]:
DOMINIOS_NOTICIAS = {
    # Internacionales
    "nytimes.com", "washingtonpost.com", "wsj.com", "ft.com",
    "theguardian.com", "bbc.com", "bbc.co.uk", "cnn.com",
    "reuters.com", "apnews.com", "bloomberg.com", "economist.com",
    "forbes.com", "time.com", "politico.com",

    # Tecnología / negocio
    "techcrunch.com", "thenextweb.com", "wired.com", "venturebeat.com",
    "arstechnica.com", "theverge.com",

    # Marketing / management (muy presentes en tu dataset)
    "blog.hubspot.com", "mitsmr.com", "hbr.org"
}
DOMINIOS_SOCIALES = {
    "twitter.com", "x.com",
    "youtube.com", "youtu.be",
    "linkedin.com",
    "instagram.com",
    "tiktok.com",
    "reddit.com",
    "facebook.com",
    "medium.com"
}
DOMINIOS_ACADEMICOS = {
    "arxiv.org",
    "nature.com",
    "science.org",
    "sciencedirect.com",
    "springer.com",
    "ieee.org",
    "acm.org",
    "journals.sagepub.com",
    "mdpi.com",
    "stanford.io"
}
DOMINIOS_EMPRESA = {
    "openai.com",
    "google.com", "ai.google", "goo.gl",
    "microsoft.com",
    "amazon.com", "aws.amazon.com",
    "meta.com",
    "ibm.com",
    "apple.com",
    "nvidia.com",
    "intel.com",
    "salesforce.com",
    "oracle.com",
    "deepmind.com"
}
DOMINIOS_DESARROLLADORES = {
    "github.com",
    "gitlab.com",
    "kaggle.com",
    "stackoverflow.com",
    "huggingface.co",
    "paperswithcode.com",
    "hubspot.com"
}
DOMINIOS_GOBIERNO = {
    "europa.eu",
    "ec.europa.eu",
    "whitehouse.gov",
    "gov.uk",
    "gov.es",
    "congress.gov",
    "oecd.org",
    "who.int"
}


In [9]:
def clasificar_fuentes(dominios):
    tipos = []

    for d in dominios:
        if d in DOMINIOS_NOTICIAS:
            tipos.append("news")
        elif d in DOMINIOS_SOCIALES:
            tipos.append("social")
        elif d in DOMINIOS_ACADEMICOS:
            tipos.append("academic")
        elif d in DOMINIOS_EMPRESA:
            tipos.append("company")
        elif d in DOMINIOS_DESARROLLADORES:
            tipos.append("developer")
        elif d in DOMINIOS_GOBIERNO:
            tipos.append("government")
        else:
            tipos.append("other")
            
    tipos_unicos = []
    vistos = set()# eliminar duplicados manteniendo orden

    for t in tipos:
        if t not in vistos:
            tipos_unicos.append(t)
            vistos.add(t)

    return tipos_unicos


Check rapido

In [10]:
df["tipo_fuente_url"] = df["dominios_url"].apply(clasificar_fuentes)
df.loc[:60, ["dominios_url", "tipo_fuente_url"]]


,dominios_url,tipo_fuente_url
0,[blog.hubspot.com],[news]
1,[bit.ly],[other]
2,[mitsmr.com],[news]
3,[bit.ly],[other]
4,[youtube.com],[social]
...,...,...
56,[venturebeat.com],[news]
57,[owler.us],[other]
58,[stanford.io],[academic]
59,[tinyurl.com],[other]


### Limpieza del tweet

In [11]:
def es_emoji(c):
    codigo = ord(c)

    if 0x1F300 <= codigo <= 0x1FAFF:
        return True
    if 0x2600 <= codigo <= 0x26FF:
        return True
    if 0x2700 <= codigo <= 0x27BF:
        return True

    return False


In [12]:
import re

def limpiar_tweet(texto):

    if type(texto) != str:
        return ""

    # 1) minúsculas
    texto = texto.lower()

    # 2) eliminar URLs
    texto = re.sub(r"http\S+|www\S+", "", texto)

    # 3) eliminar menciones
    texto = re.sub(r"@\w+", "", texto)

    # 4) convertir hashtags: "#algo" -> "algo"
    texto = re.sub(r"#(\w+)", r"\1", texto)

    # 5) construir texto conservando letras, números, espacios y EMOJIS
    resultado = []

    for c in texto:
        if c.isalnum() or c.isspace() or es_emoji(c):
            resultado.append(c)
        else:
            resultado.append(" ")

    texto = "".join(resultado)

    # 6) normalizar espacios
    texto = re.sub(r"\s+", " ", texto).strip()

    return texto


In [13]:
tweet_clean_col = []

for texto in df["tweet"]:
    texto_limpio = limpiar_tweet(texto)
    tweet_clean_col.append(texto_limpio)

df["tweet_clean"] = tweet_clean_col


In [14]:
df.loc[:5, ["tweet", "tweet_clean"]]


,tweet,tweet_clean
0,What topic could spark human dialogue on AI? A...,what topic could spark human dialogue on ai ar...
1,I think #ArtificialIntelligence will augment h...,i think artificialintelligence will augment hu...
2,What Managers Need to Know About Artificial In...,what managers need to know about artificial in...
3,4 ways Artificial Intelligence changes the gam...,4 ways artificial intelligence changes the gam...
4,Artificial Intelligence can Shape the Future o...,artificial intelligence can shape the future o...
5,#Artificialintelligence and the law #AI https...,artificialintelligence and the law ai


In [15]:
import re

def limpiar_filtrado(texto):
    if type(texto) != str:
        return ""

    texto = texto.lower()
    texto = re.sub(r"http\S+|www\S+", "", texto)
    texto = re.sub(r"@\w+", "", texto)

    # aquí sí eliminamos hashtags completos
    texto = re.sub(r"#\w+", "", texto)

    texto = re.sub(r"[^a-z0-9\s]", " ", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

tweet_clean_filt = []

for texto in df["tweet"]:
    texto_limpio = limpiar_filtrado(texto)
    tweet_clean_filt.append(texto_limpio)

df["tweet_clean_filt"] = tweet_clean_filt

In [16]:
df.loc[:6, ["tweet", "tweet_clean", "tweet_clean_filt"]]


,tweet,tweet_clean,tweet_clean_filt
0,What topic could spark human dialogue on AI? A...,what topic could spark human dialogue on ai ar...,what topic could spark human dialogue on ai ar...
1,I think #ArtificialIntelligence will augment h...,i think artificialintelligence will augment hu...,i think will augment humanity way before it di...
2,What Managers Need to Know About Artificial In...,what managers need to know about artificial in...,what managers need to know about artificial in...
3,4 ways Artificial Intelligence changes the gam...,4 ways artificial intelligence changes the gam...,4 ways artificial intelligence changes the gam...
4,Artificial Intelligence can Shape the Future o...,artificial intelligence can shape the future o...,artificial intelligence can shape the future o...
5,#Artificialintelligence and the law #AI https...,artificialintelligence and the law ai,and the law
6,IBM #Watson uses artificial intelligence to cr...,ibm watson uses artificial intelligence to cre...,ibm uses artificial intelligence to create rum...


### Filtrado de tweets
Eliminamos los tweets cuyo mensaje sea muy breve y no aporte valor

In [18]:
#contamos palabras
pd.set_option("display.max_colwidth", None)
num_palabras = []

for texto in df["tweet_clean_filt"]:
    palabras = texto.split()
    num_palabras.append(len(palabras))

df["num_palabras_filt"] = num_palabras

#eliminamos del df
filas_validas = []

for n in df["num_palabras_filt"]:
    if n >= 4:
        filas_validas.append(True)
    else:
        filas_validas.append(False)

df = df[filas_validas].copy()

#checkeamos
df.loc[:10, ["tweet", "tweet_clean", "tweet_clean_filt"]]


,tweet,tweet_clean,tweet_clean_filt
0,What topic could spark human dialogue on AI? Artificial Intelligence Glossary for Marketers https://t.co/SPyvplngtg @hubspot @Megan_Conley_,what topic could spark human dialogue on ai artificial intelligence glossary for marketers,what topic could spark human dialogue on ai artificial intelligence glossary for marketers
1,"I think #ArtificialIntelligence will augment humanity, way before it displaces it. #AI #ChasmWaxing https://t.co/SAsmyqEe2b",i think artificialintelligence will augment humanity way before it displaces it ai chasmwaxing,i think will augment humanity way before it displaces it
2,What Managers Need to Know About Artificial Intelligence https://t.co/t6VjDcGONz via @mitsmr,what managers need to know about artificial intelligence via,what managers need to know about artificial intelligence via
3,4 ways Artificial Intelligence changes the game for SEO https://t.co/uJnUtUSa2E #thenextweb,4 ways artificial intelligence changes the game for seo thenextweb,4 ways artificial intelligence changes the game for seo
4,Artificial Intelligence can Shape the Future of Society https://t.co/LOPLuWPt8W | follow @whyfuture_AI,artificial intelligence can shape the future of society follow,artificial intelligence can shape the future of society follow
6,IBM #Watson uses artificial intelligence to create rum that tastes like a holiday https://t.co/XtxV3aYADa https://t.co/Lg8Lc0KOGX,ibm watson uses artificial intelligence to create rum that tastes like a holiday,ibm uses artificial intelligence to create rum that tastes like a holiday
7,How Artificial Intelligence Will Revolutionize eDiscovery https://t.co/elT5C0xm0N https://t.co/OvoqxjkvgK,how artificial intelligence will revolutionize ediscovery,how artificial intelligence will revolutionize ediscovery
8,How Artificial Intelligence Will Kickstart the Internet of Things #Nitaipartners #internetofthings #bigdataanalytic… https://t.co/Hcp6VPQF4C,how artificial intelligence will kickstart the internet of things nitaipartners internetofthings bigdataanalytic,how artificial intelligence will kickstart the internet of things
9,This man is not mentally all there. That or he's actually artificial intelligence that's not that intelligent. He makes C-3PO look smart.🙄😒 https://t.co/42m4kGG3NY,this man is not mentally all there that or he s actually artificial intelligence that s not that intelligent he makes c 3po look smart 🙄😒,this man is not mentally all there that or he s actually artificial intelligence that s not that intelligent he makes c 3po look smart
10,Modular housing and artificial intelligence among ideas recognized at STEM awards https://t.co/RDObQjMjAE,modular housing and artificial intelligence among ideas recognized at stem awards,modular housing and artificial intelligence among ideas recognized at stem awards


### Guardamos texto preprocesado 

In [19]:
df.to_parquet("../data/processed/tweets_ai_clean.parquet", index=False)